# Day 31 — **Observer Pattern: Agent Event Bus**

**Phase 2 · Module 5: Multi-Agent Orchestration** · ~30 min

Phase 2 starts today, and the first problem multi-agent systems hit is *wiring*. Four agents, and each one needs logging, metrics, an audit trail, and a UI update. Call all four from every agent and you have sixteen hard-coded couplings — change the logger and you touch every agent.

The **Observer** pattern inverts it: agents *publish* facts about what happened and don't care who listens. Subscribers register themselves. That's the event bus behind LangGraph's `astream_events`, CrewAI's callbacks, and every trace exporter you'll use.

Today:
1. Events as immutable **data**.
2. A minimal **EventBus** — `subscribe` / `publish`.
3. Real subscribers: **logging** + **metrics** on one agent run.
4. **Fault isolation** and async handlers — one bad subscriber must not kill the agent.

## 1. Events are data, not function calls

An event is a fact that already happened: a type, a payload, a timestamp, and the `trace_id` that ties it to a request. Make it `frozen` — a subscriber must never mutate what another subscriber sees.

In [1]:
from dataclasses import dataclass, field
from datetime import datetime, timezone
from typing import Any

@dataclass(frozen=True, slots=True)
class AgentEvent:
    """One immutable fact emitted by an agent."""
    type: str
    trace_id: str
    payload: dict[str, Any] = field(default_factory=dict)
    ts: datetime = field(default_factory=lambda: datetime.now(timezone.utc))

e = AgentEvent("task_started", trace_id="req-7781", payload={"agent": "kyc_checker"})
print(e.type, "|", e.trace_id, "|", e.payload)

try:
    e.type = "hacked"
except Exception as exc:
    print("frozen ->", type(exc).__name__, exc)

task_started | req-7781 | {'agent': 'kyc_checker'}
frozen -> FrozenInstanceError cannot assign to field 'type'


☝ `frozen=True` gives you two things at once: subscribers can't corrupt each other's view of an event, and the event is hashable, so it can be de-duplicated or cached. `slots=True` drops the per-instance `__dict__` — worth having when a busy agent emits thousands of these per minute.

## 2. The bus — `subscribe(event_type, handler)` and `publish(event)`

Twenty lines. A dict from event type to handlers, plus a `"*"` wildcard for subscribers that want everything (tracing, audit).

In [2]:
from collections import defaultdict
from typing import Callable

class EventBus:
    """Minimal synchronous publish/subscribe bus."""

    def __init__(self) -> None:
        self._subs: dict[str, list[Callable[[AgentEvent], None]]] = defaultdict(list)

    def subscribe(self, event_type: str, handler: Callable[[AgentEvent], None]) -> None:
        self._subs[event_type].append(handler)

    def publish(self, event: AgentEvent) -> None:
        for handler in [*self._subs[event.type], *self._subs["*"]]:
            handler(event)

bus = EventBus()
bus.subscribe("task_started", lambda ev: print(f"  [started] {ev.payload['agent']}"))
bus.subscribe("*",            lambda ev: print(f"  [audit  ] {ev.type} ({ev.trace_id})"))

bus.publish(AgentEvent("task_started", "req-7781", {"agent": "kyc_checker"}))
bus.publish(AgentEvent("task_completed", "req-7781", {"agent": "kyc_checker"}))

  [started] kyc_checker
  [audit  ] task_started (req-7781)
  [audit  ] task_completed (req-7781)


☝ Note what the publisher does *not* know: how many subscribers exist, what they do, or whether any exist at all. That's the decoupling. `"*"` is the hook that makes cross-cutting concerns — audit logging in a bank is one — a single registration rather than an edit to every agent.

## 3. Two real subscribers: logging + metrics

Now the useful part. One agent publishes `task_started`, `tool_called`, `task_completed`. A **logging** handler renders lines; a **metrics** handler counts and times. Neither is mentioned by the agent's code.

In [3]:
import time

class MetricsHandler:
    """Counts events and measures task latency per agent."""

    def __init__(self) -> None:
        self.counts: dict[str, int] = defaultdict(int)
        self._open: dict[str, float] = {}
        self.latency_ms: dict[str, float] = {}

    def __call__(self, ev: AgentEvent) -> None:
        self.counts[ev.type] += 1
        if ev.type == "task_started":
            self._open[ev.trace_id] = time.perf_counter()
        elif ev.type == "task_completed":
            started = self._open.pop(ev.trace_id, None)
            if started is not None:
                self.latency_ms[ev.trace_id] = (time.perf_counter() - started) * 1000

def log_handler(ev: AgentEvent) -> None:
    print(f'{ev.ts:%H:%M:%S} {ev.type:<15} trace={ev.trace_id} {ev.payload}')

metrics = MetricsHandler()
bus = EventBus()
bus.subscribe("*", log_handler)
bus.subscribe("*", metrics)

def kyc_agent(customer: str, trace_id: str, bus: EventBus) -> str:
    """A tiny agent that narrates itself onto the bus."""
    bus.publish(AgentEvent("task_started", trace_id, {"agent": "kyc_checker", "customer": customer}))
    bus.publish(AgentEvent("tool_called", trace_id, {"tool": "sanctions_list", "hit": False}))
    bus.publish(AgentEvent("tool_called", trace_id, {"tool": "pep_screen", "hit": False}))
    verdict = "CLEAR"
    bus.publish(AgentEvent("task_completed", trace_id, {"agent": "kyc_checker", "verdict": verdict}))
    return verdict

print(kyc_agent("ACME Ltd", "req-7781", bus), "\n")
print("counts   :", dict(metrics.counts))
print("latency  :", {k: f"{v:.2f} ms" for k, v in metrics.latency_ms.items()})

15:00:44 task_started    trace=req-7781 {'agent': 'kyc_checker', 'customer': 'ACME Ltd'}
15:00:44 tool_called     trace=req-7781 {'tool': 'sanctions_list', 'hit': False}
15:00:44 tool_called     trace=req-7781 {'tool': 'pep_screen', 'hit': False}
15:00:44 task_completed  trace=req-7781 {'agent': 'kyc_checker', 'verdict': 'CLEAR'}
CLEAR 

counts   : {'task_started': 1, 'tool_called': 2, 'task_completed': 1}
latency  : {'req-7781': '0.06 ms'}


☝ `kyc_agent` has no logger and no metrics client — it publishes four facts. Everything else is subscriber configuration, which means the same agent runs unchanged in a notebook (print handler), in CI (assert-on-events handler), and in production (CloudWatch + OpenTelemetry handlers). Swapping observability stacks becomes a wiring change, not a code change across every agent.

## 4. Fault isolation — a broken subscriber must not kill the agent

The naive `publish` above has a real bug: one handler raising takes down the publisher. In a multi-agent system that means a flaky metrics exporter fails a customer's loan check. Isolate each handler, and support `async` ones while you're there.

In [4]:
import asyncio, inspect

class AsyncEventBus:
    """Publish/subscribe bus that isolates handler failures and supports async handlers."""

    def __init__(self) -> None:
        self._subs: dict[str, list[Callable]] = defaultdict(list)
        self.handler_errors: list[tuple[str, str]] = []

    def subscribe(self, event_type: str, handler: Callable) -> None:
        self._subs[event_type].append(handler)

    async def publish(self, event: AgentEvent) -> None:
        handlers = [*self._subs[event.type], *self._subs["*"]]

        async def run(h):
            result = h(event)
            if inspect.isawaitable(result):
                await result

        results = await asyncio.gather(*(run(h) for h in handlers), return_exceptions=True)
        for h, r in zip(handlers, results):
            if isinstance(r, Exception):
                name = getattr(h, "__name__", type(h).__name__)
                self.handler_errors.append((name, f"{type(r).__name__}: {r}"))

def flaky_exporter(ev: AgentEvent) -> None:
    raise ConnectionError("metrics endpoint unreachable")

async def slow_audit(ev: AgentEvent) -> None:
    await asyncio.sleep(0.01)
    print(f"  [audit ] persisted {ev.type}")

abus = AsyncEventBus()
abus.subscribe("*", flaky_exporter)
abus.subscribe("*", slow_audit)
abus.subscribe("*", lambda ev: print(f"  [log   ] {ev.type}"))

await abus.publish(AgentEvent("task_completed", "req-9002", {"verdict": "CLEAR"}))
print("\nagent survived. handler errors:", abus.handler_errors)

  [log   ] task_completed
  [audit ] persisted task_completed

agent survived. handler errors: [('flaky_exporter', 'ConnectionError: metrics endpoint unreachable')]


☝ `return_exceptions=True` is the whole trick (Day 09's `asyncio.gather` lesson, now load-bearing): the failing exporter is recorded, the audit and log handlers still ran, and the agent returned normally. The rule for production buses — **a subscriber failure is a subscriber's problem**. Surface it in `handler_errors` and alert on it; never let it propagate back into the agent that published.

## 5. Recap — publish facts, subscribe behaviour

| Concern | Without a bus | With the bus |
|---|---|---|
| Add metrics to 4 agents | edit 4 agents | 1 `subscribe("*", metrics)` |
| Swap log sink | edit every agent | swap one handler |
| Audit trail (regulatory) | scattered calls | one wildcard subscriber |
| Handler crashes | agent dies | recorded in `handler_errors` |

An event bus is what makes a multi-agent system *observable without being coupled*: agents narrate `task_started` / `tool_called` / `task_completed`, and logging, metrics, audit and UI are all just listeners. Keep events **immutable dataclasses**, always carry a `trace_id`, and always isolate handler failures.

You'll see the same shape everywhere in Phase 2 — LangGraph's `astream_events`, CrewAI's step callbacks, and OpenTelemetry span exporters are all this pattern at production scale. Today's module builds the topologies that emit these events.